In [1]:
from datetime import datetime

from demo_flow import call_llm, call_llm_2, flow, PromptUser, call_llm_3

with open(r"prompts\v5.md", encoding="utf-8") as f:
    sys = f.read()

conversation_history = []
action_log = []
action_taken = False
no_response_counter = 0
start = False
res = ""
while True:
    if not action_taken and start:
        user_msg = input(res)
        if user_msg == "q":
            break
        conversation_history.append({"message": user_msg, "role": "user", "timestamp": datetime.now()})
        
    elif not start:
        conversation_history.append({"message": "<Call Started>", "role": "user", "timestamp": datetime.now()})
        start = True

    agent_output = await call_llm_3(sys, flow, conversation_history, action_log, "gpt-4.1")
    if agent_output is None:
        break
    if not isinstance(agent_output.next_action, PromptUser):
        action_taken = True
        no_response_counter += 1
        # if no_response_counter >= 3:
        #     event_stream.append("[SYSTEM: You have not responded to the user for three consecutive actions, give them a progress cue]")
    else:
        # reset
        res = agent_output.next_action.prompt
        no_response_counter = 0
        action_taken = False


<conversation_history>
[User, just now]: <Call Started>
</conversation_history>

<agent_script>
<action_log>
No Actions Taken Yet
</action_log>

<current_node>
Current Path: Call Start

Node: Call Start
Is Terminal Node?: False
Instructions: Greet the user, Inroduce yourself, then Learn what kind of service does the user need, raise a repair ticket, or update an existing one.
---

Service Type (Radio Field)
( ) Raise repair ticket
( ) Update existing ticket
</current_node>
</agent_script>
ParsedChatCompletionMessage[call_llm_3.<locals>.AgentOutput](content='{"thoughts":"It\'s the start of the conversation. I need to introduce myself, greet the caller, and ask which service they need: raising a new repair or updating an existing one.","next_action":{"prompt":"Hello, thanks for calling Midland Heart repairs. You\'re speaking to the Virtual Repairs Assistant. Are you calling to report a new repair or to get an update on an existing repair?"}}', refusal=None, role='assistant', annotations=

In [1]:
from demo_flow import flow

In [3]:
flow.current_action_model().model_json_schema()

{'$defs': {'SelectServiceType': {'description': "Selects an option for radio field 'Service Type' and reflects it to the agent script current node.",
   'properties': {'field_label': {'const': 'Service Type',
     'default': 'Service Type',
     'title': 'Field Label',
     'type': 'string'},
    'value': {'description': 'option to select',
     'enum': ['Raise repair ticket', 'Update existing ticket'],
     'title': 'Value',
     'type': 'string'}},
   'required': ['value'],
   'title': 'SelectServiceType',
   'type': 'object'}},
 'properties': {'update': {'default': None,
   'description': 'short ~3-5 word update for the user',
   'title': 'Update',
   'type': 'string'},
  'action': {'description': 'Data field action to take',
   'items': {'$ref': '#/$defs/SelectServiceType'},
   'title': 'Action',
   'type': 'array'}},
 'required': ['action'],
 'title': 'DataFieldAction',
 'type': 'object'}

In [2]:
from typing import Union

list[Union[str, int]]

list[typing.Union[str, int]]

In [1]:
from pydantic import BaseModel

class Test(BaseModel):
    b: str
    a: str

test = Test(a="hello", b="bye")

In [4]:
test.__class__.__name__

'Test'

In [ ]:
for a in test:
    print(a)

In [2]:
flow.current_action_model()

typing.Union[list[wizard.SelectCouldYouFitAOneEuroCoinInTheGap, wizard.UpdateUser], wizard.PromptUser, wizard.GoBack, wizard.GoToNode]

In [3]:
from pydantic import BaseModel, create_model, Field

class Dog(BaseModel):
    "A nice lil doggo called lilo"
    bark: int = Field(..., description="bark power")
    name: str

class Cat(BaseModel):
    "cute kitty called bobby"
    meow: int = Field(..., description="meow power")
    name: str


class PetOwner(BaseModel):
    pet: Dog | Cat = Field(..., description="pet of choice")

In [4]:
Dog = create_model(
    "Dog",
    bark = (int, Field(..., description="test desc")),
    test="test",
    __doc__="hello world"

)

PydanticUserError: A non-annotated attribute was detected: `test = 'test'`. All model fields require a type annotation; if `test` is not meant to be a field, you may be able to resolve this error by annotating it as a `ClassVar` or updating `model_config['ignored_types']`.

For further information visit https://errors.pydantic.dev/2.10/u/model-field-missing-annotation

In [6]:
from pydantic import BaseModel

class Ans(BaseModel):
    numbers: list[str, int]

In [7]:
Ans.model_json_schema()

{'properties': {'numbers': {'items': {'type': 'string'},
   'title': 'Numbers',
   'type': 'array'}},
 'required': ['numbers'],
 'title': 'Ans',
 'type': 'object'}

In [19]:
import os

from dotenv import load_dotenv

load_dotenv()

import openai

client = openai.AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])
res = await client.beta.chat.completions.parse(
                model="gpt-4o",
                messages=[
                    {
                        "role": "user",
                        "content": "return one single number",
                    },
                ],
                response_format=Ans,
            )

In [20]:
res.choices[0].message.parsed

Ans(numbers=7)

In [ ]:
res.choices[0].message.parsed.model_dump_json()

'{"pet":{"bark":5,"name":"Lilo"}}'

In [ ]:
flow.ctx